# Explore tomogram data and features

Inspect preprocessed tomograms, visualize 2D slices, and explore the extracted feature space.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path('../data/processed')  # adjust to your path
runs = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print(f'Runs: {runs}')

In [ ]:
# Visualize 3 orthogonal slices from one tomogram
run = runs[0]
tomo = np.load(DATA_DIR / run / 'tomogram.npy')
print(f'{run}: shape={tomo.shape}  mean={tomo.mean():.3f}  std={tomo.std():.3f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
z, y, x = [s // 2 for s in tomo.shape]
for ax, sl, title in zip(axes, [tomo[z], tomo[:, y, :], tomo[:, :, x]],
                          [f'XY (Z={z})', f'XZ (Y={y})', f'YZ (X={x})']):
    ax.imshow(sl, cmap='gray')
    ax.set_title(f'{run} — {title}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Load and display the labels CSV
labels_path = Path('../labels.csv')
if labels_path.exists():
    df = pd.read_csv(labels_path)
    print(df.to_string())
    print(f'\nFeature counts:')
    for col in df.columns[1:]:
        n = df[col].astype(str).str.strip().replace({'1': True, '0': False}).sum()
        print(f'  {col}: {n}/{len(df)} positive')
else:
    print('labels.csv not found — run label_tomograms.py first')

In [ ]:
# Visualize feature space with PCA (after running extract_features.py)
features_path = Path('../features.npz')
if features_path.exists() and labels_path.exists():
    from sklearn.decomposition import PCA
    data = np.load(features_path, allow_pickle=True)
    X, run_names = data['X'], list(data['run_names'])
    df = pd.read_csv(labels_path).set_index('tomogram_id')
    
    pca = PCA(n_components=2)
    X2 = pca.fit_transform(X)
    
    features = [c for c in df.columns]
    fig, axes = plt.subplots(1, len(features), figsize=(5*len(features), 4))
    if len(features) == 1:
        axes = [axes]
    
    for ax, feat in zip(axes, features):
        colors = []
        for rn in run_names:
            if rn in df.index:
                val = str(df.loc[rn, feat]).strip()
                colors.append('red' if val == '1' else 'blue' if val == '0' else 'gray')
            else:
                colors.append('gray')
        ax.scatter(X2[:, 0], X2[:, 1], c=colors, alpha=0.7)
        ax.set_title(f'PCA: {feat}\nred=present, blue=absent')
    plt.tight_layout()
    plt.show()
else:
    print('Run extract_features.py first to generate features.npz')